# Aralin 08 - Multi-Agent Design Pattern


## Setup


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Bakit Multi-Agent Systems?

Ang mga totoong gawain tulad ng pagpaplano ng biyahe ay nangangailangan ng iba't ibang uri ng kadalubhasaan — logistics, lokal na kaalaman, pagtatakda ng badyet, at iba pa. Ang isang solong ahente na nagsusubukang hawakan ang lahat ay mabilis na nagiging mahirap kontrolin.

Nilulutas ito ng mga multi-agent systems sa pamamagitan ng **espesyalisasyon**: bawat ahente ay nakatuon sa isang larangan ng kadalubhasaan, na nagreresulta sa mas mataas na kalidad kaysa sa isang generalist. Pinapabuti rin nila ang **scalability** — maaari kang magdagdag ng mga bagong ahente (hal., isang flight specialist, isang restaurant critic) nang hindi nire-rewrite ang umiiral na workflow. Ang mga ahente ay nagkukumpone sa pamamagitan ng isang istrakturadong pipeline, na nagpapasa ng konteksto mula sa isa patungo sa susunod.


## Paglikha ng mga Espesyalisadong Ahente


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Paggawa ng Isang Sunud-sunod na Daloy ng Trabaho

Pinapayagan ka ng `WorkflowBuilder` na iugnay ang mga ahente sa isang direktadong grapika. Dito tayo gumawa ng isang simpleng dalawang-hakbang na pipeline: ang **TravelPlanner** ang gumagawa ng itineraryo, pagkatapos ay nire-review at pinapaganda ito ng **TravelConcierge**.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Pagdaragdag ng Higit pang mga Ahente sa Daloy ng Trabaho

Isa sa pinakamalaking bentahe ng multi-agent pattern ay kung gaano ito kadaling palawakin. Sa ibaba ay nagdagdag tayo ng isang **BudgetReviewer** na ahente na sumusuri sa plano laban sa badyet ng biyahero, nagtatalaga ng mga item na maaaring magdulot ng labis na gastos, at nagmumungkahi ng mga alternatibong makatipid ng pera. Ang daloy ng trabaho ay ngayon nagpapatakbo ng tatlong ahente nang sunod-sunod:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Buod

Sa araling ito natutunan mo kung paano:

1. **Lumikha ng mga espesyal na ahente** — bawat isa ay may pokus na papel (pagpaplano, concierge, pagsusuri ng badyet).
2. **Ikabit ang mga ahente sa isang sunud-sunod na workflow** gamit ang `WorkflowBuilder` at `add_edge`.
3. **I-stream ang output** mula sa isang multi-agent pipeline, sinusubaybayan kung aling ahente ang nagsasalita.
4. **Palawakin ang isang workflow** sa pamamagitan ng pagdagdag ng mga bagong ahente sa kadena nang hindi binabago ang mga kasalukuyang ahente.

Pinananatiling simple ng multi-agent na disenyo ang bawat ahente habang lumilikha ng mas mayaman, mas masusing na-review na mga resulta kaysa sa kahit anong makakamit ng isang ahente lamang nang mag-isa.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
